# LeetCode #1240: Tiling a Rectangle with the Fewest Squares

https://leetcode.com/problems/tiling-a-rectangle-with-the-fewest-squares/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(\min(m,n)^{mn})$ | $O(mn)$ |
| **Optimal: Backtracking + Skyline Pruning ★** | $O(\min(m,n)^{mn})$ (pruned) | $O(mn)$ |

---

## Understanding the Methods

### Brute Force
Try every possible square at every empty cell in every size, recursively filling the rectangle. Without pruning the search space is astronomical and takes forever even for moderate inputs.

### Optimal: Backtracking + Skyline Pruning ★
Maintain a "skyline" array `h[j]` = current filled height at column `j`. Always fill the lowest column. Place a square of side `s` (from largest possible down to 1) starting at that column, update the skyline, recurse, then backtrack. Prune branches where the current square count already matches or exceeds the known best.

**Why this is better than Brute Force:** The greedy fill order (lowest column first) eliminates redundant orderings, and the `count >= best` pruning cuts vast branches. For the hardest known case (11×13) it reaches 6 squares quickly.

**Constraints:**
* $1 \leq n, m \leq 13$

## Solutions

### C#

In [ ]:
public class Solution {
    int best;

    public int TilingRectangle(int n, int m) {
        best = n * m; // Worst case: fill with 1×1 squares
        Backtrack(new int[m], n, m, 0);
        return best;
    }

    void Backtrack(int[] sky, int n, int m, int count) {
        // Prune: already worse than best known solution
        if (count >= best) return;

        // Find the column with the lowest skyline — we must fill it next
        int minH = sky[0], col = 0;
        for (int j = 1; j < m; j++) {
            if (sky[j] < minH) { minH = sky[j]; col = j; }
        }

        // All columns filled — a complete tiling found
        if (minH == n) { best = count; return; }

        // Determine maximum square side that fits at (minH, col)
        int maxSide = Math.Min(n - minH, m - col);
        for (int j = col + 1; j < m && sky[j] <= minH; j++) { /* extend rightward */ }
        // Limit side by the contiguous flat region starting at col
        int flat = 1;
        while (col + flat < m && sky[col + flat] <= minH) flat++;
        maxSide = Math.Min(maxSide, flat);

        // Try each square size from largest to smallest to reach good solutions fast
        for (int s = maxSide; s >= 1; s--) {
            // Place square of side s at (col, minH)
            for (int j = col; j < col + s; j++) sky[j] += s;
            Backtrack(sky, n, m, count + 1);
            // Undo placement
            for (int j = col; j < col + s; j++) sky[j] -= s;
        }
    }
}

### Python

In [ ]:
class Solution:
    def tiling_rectangle(self, n: int, m: int) -> int:
        self.best = n * m  # Worst case: fill with 1x1 squares

        def backtrack(sky: list, count: int) -> None:
            # Prune: already worse than best known solution
            if count >= self.best:
                return

            # Find the column with the lowest skyline — we must fill it next
            min_h = min(sky)
            col = sky.index(min_h)

            # All columns filled — a complete tiling found
            if min_h == n:
                self.best = count
                return

            # Limit side by contiguous flat region and remaining height/width
            flat = 0
            while col + flat < m and sky[col + flat] <= min_h:
                flat += 1
            max_side = min(n - min_h, flat)

            # Try each square size from largest to smallest
            for s in range(max_side, 0, -1):
                for j in range(col, col + s):
                    sky[j] += s
                backtrack(sky, count + 1)
                for j in range(col, col + s):
                    sky[j] -= s

        backtrack([0] * m, 0)
        return self.best

### Go

In [ ]:
func tilingRectangle(n int, m int) int {
    best := n * m // Worst case: fill with 1x1 squares
    sky := make([]int, m)

    var backtrack func(count int)
    backtrack = func(count int) {
        // Prune: already worse than best known solution
        if count >= best {
            return
        }
        // Find the column with the lowest skyline — we must fill it next
        minH, col := sky[0], 0
        for j := 1; j < m; j++ {
            if sky[j] < minH {
                minH, col = sky[j], j
            }
        }
        // All columns filled — a complete tiling found
        if minH == n {
            best = count
            return
        }
        // Limit side by contiguous flat region and remaining space
        flat := 0
        for col+flat < m && sky[col+flat] <= minH {
            flat++
        }
        maxSide := n - minH
        if flat < maxSide {
            maxSide = flat
        }
        // Try each square size from largest to smallest
        for s := maxSide; s >= 1; s-- {
            for j := col; j < col+s; j++ {
                sky[j] += s
            }
            backtrack(count + 1)
            for j := col; j < col+s; j++ {
                sky[j] -= s
            }
        }
    }
    backtrack(0)
    return best
}

### Rust

In [ ]:
impl Solution {
    pub fn tiling_rectangle(n: i32, m: i32) -> i32 {
        let (n, m) = (n as usize, m as usize);
        let mut best = n * m; // Worst case: fill with 1x1 squares
        let mut sky = vec![0usize; m];

        fn backtrack(sky: &mut Vec<usize>, n: usize, m: usize, count: usize, best: &mut usize) {
            // Prune: already worse than best known solution
            if count >= *best { return; }
            // Find the column with the lowest skyline — we must fill it next
            let min_h = *sky.iter().min().unwrap();
            let col = sky.iter().position(|&h| h == min_h).unwrap();
            // All columns filled — a complete tiling found
            if min_h == n { *best = count; return; }
            // Limit side by contiguous flat region and remaining space
            let flat = sky[col..].iter().take_while(|&&h| h <= min_h).count();
            let max_side = (n - min_h).min(flat);
            // Try each square size from largest to smallest
            for s in (1..=max_side).rev() {
                for j in col..col+s { sky[j] += s; }
                backtrack(sky, n, m, count + 1, best);
                for j in col..col+s { sky[j] -= s; }
            }
        }

        backtrack(&mut sky, n, m, 0, &mut best);
        best as i32
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `n = 2, m = 3`
One 2×2 square fills columns 0–1, one 1×1 fills the remaining cell. Answer: **3** (one 2×2 + two 1×1 covering the rest). Actually the minimum is 3: one 2×2 at (0,0) + two 1×1 at column 2. ✓

### 2. Slightly Complex
**Input:** `n = 5, m = 8`
Since $5 \nmid 8$ and $8 \nmid 5$, neither pure row/column tiling works. The backtracking finds the optimal arrangement using **5** squares (one 5×5 + four 1×1 is suboptimal; the true optimum requires creative placement).

### 3. Edge Case: Time Factor
**Input:** `n = 11, m = 13`
The hardest known input — greedy gives 8 squares but the true optimum is **6**. The backtracking must explore many branches and is the worst-case scenario for this $n, m$ range.

### 4. Edge Case: Space Factor
**Input:** `n = 13, m = 13`
The skyline array has 13 entries; the recursion depth equals the number of squares placed ($\leq 169$ 1×1s). Stack depth is $O(mn)$ in the degenerate case.

### 5. Almost-Impossible but Plausible
**Input:** `n = 13, m = 1`
A 13×1 rectangle needs exactly **13** unit squares — no larger square fits in the width-1 strip. The algorithm immediately restricts `maxSide = 1` at every step, finding the answer in 13 recursive calls.